# Demo: Data Access with Provider Gateway

This notebook demonstrates the new approach to data access using the `provider` gateway. With a single initialization, you can access experiment data as DataFrames for analysis.

**Key Features:**

1. **Database Location:** Place your DuckDB database file (e.g., `DB.duckdb`) in the `data/` directory. The provider automatically looks for the database there during initialization. 
2. **Schema Inspection:** Use `provider.schema()` to display the table structure, including all available columns and their types.
3. **Extending with Custom Queries:** You can add new queries to the loader.py and make them accessible via provider.py they are then available with a simple proviver.<query_name>() call. 
4. **Safe Connection Handling:** The provider manages the database connection for you. If you need to close the connection manually (e.g., at the end of a script), call `provider.close()`. 

- Initialization is one line: `provider.init()`
- Access any table or query as a DataFrame: for example
  `provider.df()`, `provider.plates()`, `provider.slots()`, `provider.signals()`, `provider.data_origin()`, `provider.plate_slots()`, `provider.grouped_data()`

**Main components of the data access layer**
- The `src/loader.py` contains the actual queries and logic
- The `src/provider.py` provides the interface to access the data and manages the connection between the notebooks and the database.

- the `src/data_processing.py` performs data transformations and processing steps, which can called anywhere in the notebooks

- the `viz/` folder contains visualization utilities that can be used to create plots and interactive widgets for the notebooks


In [ ]:
# 1. Environment and Provider Initialization
%load_ext autoreload
%autoreload 2
%run ../config/preamble.py

from src import provider

# Initialize the provider gateway
provider.init()

## Choosing and switching the database

By default `provider.init()` **auto-detects** a single `.duckdb` file in the `data/` directory. If **more than one** database is present, it selects the **most recently modified** one. You can check which database is currently active with `provider.where()`.

To pick a specific database explicitly, use `db_path`. The easiest way is to feed a path from `list_databases()` straight into `init()` — it returns **absolute paths**, so this works regardless of the notebook's working directory:

```python
# See which databases are available (works before init(), newest first, absolute paths)
provider.list_databases()          # -> ['/.../data/DB.duckdb', '/.../data/OldDB.duckdb']

# Select one explicitly
provider.init(db_path=provider.list_databases()[0])

# Or pass any path yourself
provider.init(db_path="../data/DB.duckdb")
```

**Switching database mid-session:** just call `init()` again with a different `db_path`. It is cleanest to close the current connection first:

```python
provider.close()
provider.init(db_path=provider.list_databases()[1])
provider.where()                   # confirm which DB + table is now active
```

In [ ]:
# List available databases, then (re)initialize on a chosen one by name
provider.list_databases()

##  Data exploration 

After initialization, you can first explore the database 

In [ ]:
# Show the schema of the current table
provider.schema()

In [ ]:
# List all available plates
provider.plates()

In [ ]:
# List all slots for a specific plate (replace 'Plate_1' with a real plate name)
provider.slots(22)

In [ ]:
# List all (plate, slot) pairs
provider.plate_slots()

In [ ]:
# List all signals for a plate and slot (replace with real values)
signals = provider.signals(22, 1)
signals

In [ ]:
# List all data origins for a plate and slot 
provider.data_origin(22, 1)

## DataFrames
You can load a subset of data as a DataFrame using provider.df()\
Use these options to flexibly select, filter, and limit the data you load as a DataFrame for your analysis.

* plate (required) and slot,
* columns to select,
* signal names,
* data origin,
* WCS_Y_mm value range,
* row limit,
* and whether to order by time.
* and other columns that you can check in the schema.

In [ ]:
# 1. Get all data for a specific plate and slot
df = provider.df(22)
df.sample(15)


In [ ]:
# 2. Get all current measurements (in amps) for plate 22, slot 1, only from HF_Data
df = provider.df(22, 1, signals=["CURRENT|4"], data_origin="HF_Data", fields=["Time", "Value", "Unit"])
df.head(10)

In [ ]:
# 3. Get all contour deviation values 
df = provider.df(22, 1, signals=["CONT_DEV|6"], fields=["Time", "Value", "Description"])

df = df[df["Value"] > 0]
if not df.empty:
    display(df.sample(15))
else:
    print("No rows with Value > 0.")

In [ ]:
# 4. Get all WCSPosition values for plate 22, slot 1, in a specific time range
df = provider.df(24, 1, signals=["WCSPosition"], fields=["Time", "Value"])
df.sample(20)

In [ ]:
# 5. Get all data for plate 22, slot 1, where Axis is 'A'
df = provider.df(22, 1, fields=["Time", "Signal", "Value", "Axis"])
df = df[df["Axis"] == "A"]
df.head(15)

In [ ]:
# 6. Get the first 50 rows of all signals for plate 22, slot 1, ordered by time
df = provider.df(22, 1, limit=50, order_by_time=True)
df.head(15)

## GROUP_DATA

You can use provider.group_data() to group by one or more columns (such as Signal, DataOrigin, or Axis) and apply aggregation functions: mean, sum, count, min, or max to selected columns.

In [ ]:
# 1. Mean value per signal for a plate and slot
df = provider.group_data(group_by=["Signal"], agg={"Value": "mean"}, plate=22, slot=1)
df.head()

In [ ]:
# 2. Maximum value per DataOrigin
df = provider.group_data(group_by=["DataOrigin"], agg={"Value": "max"}, plate=22, slot=1)
df.head()

In [ ]:
# 3. Count of records per Axis
df = provider.group_data(group_by=["Axis"], agg={"Value": "count"}, plate=22, slot=1)
df.head()

In [ ]:
# 4. Sum of Value per Signal and DataOrigin
df = provider.group_data(group_by=["Signal", "DataOrigin"], agg={"Value": "sum"}, plate=22, slot=1)
df.head()

## Provider in other scripts

The provider module can also be used in your data processing scripts or modules, not just in notebooks. For example, you can import provider in your data_processing.py and use provider.df() or provider.group_data() to load and process data in the same way as in your notebook.

In [ ]:
# Import  the module 
from src import provider

# 1. Environment and Provider Initialization
%load_ext autoreload
%autoreload 1
%run ../config/preamble.py
provider.init()

# Now you can process df as needed in your scripts
selected_df = provider.df(22, 1, fields=["Time", "Value"])


## Custom SQL queries
you can run **your own SQL** through the loader with two public helpers. This keeps the DuckDB connection encapsulated in the loader — you author the SQL, the loader executes it:

- `provider.query_df(sql, params=...)` → returns a **DataFrame**
- `provider.query_row(sql, params=...)` → returns the **first row** as a tuple (or `None`)

Use `?` placeholders with `params=[...]` for values (never string-format them in — this is safe against injection and type issues). `provider.table()` returns the connected table name so the SQL stays portable across databases.


In [ ]:
# query_df: custom aggregation as a DataFrame
table = provider.table()   # connected table name -> portable SQL

sql = f"""
    SELECT Signal, COUNT(*) AS n_rows, AVG(Value) AS avg_value
    FROM {table}
    WHERE Platte = ? AND DataOrigin = ?
    GROUP BY Signal
    ORDER BY n_rows DESC
"""
provider.query_df(sql, params=["22", "HF_Data"]).head(10)

In [ ]:
# query_row: single row / scalars as a tuple (or None)
table = provider.table()

sql = f"SELECT MIN(Value) AS min_value, MAX(Value) AS max_value FROM {table} WHERE Platte = ?"
row = provider.query_row(sql, params=["22"])
print("min, max Value for plate 22:", row)

## WIDGETS in notebooks ##
There is an easy way to filter and select dataframes in the notebooks with interactive widgets.\
Plate is required and you can select all the other values to filter the dataframe. The widgets are in the `viz/widgets.py` and can be used in any notebook.

In [ ]:
from viz.widgets import show_plate_slot_filter_widget
show_plate_slot_filter_widget()